###1) Konfiguracja i kontrola źródeł

In [0]:
from datetime import date, datetime, time, timezone
from zoneinfo import ZoneInfo

from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text(
    "evaluation_start_date_local",
    "2024-07-01"
)
dbutils.widgets.text(
    "train_end_date_local_exclusive",
    "2024-09-01"
)
dbutils.widgets.text(
    "validation_end_date_local_exclusive",
    "2024-10-01"
)
dbutils.widgets.text(
    "evaluation_end_date_local_exclusive",
    "2024-11-01"
)

EVALUATION_START_DATE_TEXT = dbutils.widgets.get(
    "evaluation_start_date_local"
)
TRAIN_END_DATE_TEXT = dbutils.widgets.get(
    "train_end_date_local_exclusive"
)
VALIDATION_END_DATE_TEXT = dbutils.widgets.get(
    "validation_end_date_local_exclusive"
)
EVALUATION_END_DATE_TEXT = dbutils.widgets.get(
    "evaluation_end_date_local_exclusive"
)

MARKET_TIMEZONE = "America/Los_Angeles"
LOCAL_TIMEZONE = ZoneInfo(MARKET_TIMEZONE)

evaluation_start_local = datetime.combine(
    date.fromisoformat(EVALUATION_START_DATE_TEXT),
    time.min,
    tzinfo=LOCAL_TIMEZONE,
)
evaluation_end_local = datetime.combine(
    date.fromisoformat(EVALUATION_END_DATE_TEXT),
    time.min,
    tzinfo=LOCAL_TIMEZONE,
)

EVALUATION_START_UTC = evaluation_start_local.astimezone(
    timezone.utc
)
EVALUATION_END_UTC = evaluation_end_local.astimezone(
    timezone.utc
)

if not (
    date.fromisoformat(EVALUATION_START_DATE_TEXT)
    < date.fromisoformat(TRAIN_END_DATE_TEXT)
    < date.fromisoformat(VALIDATION_END_DATE_TEXT)
    < date.fromisoformat(EVALUATION_END_DATE_TEXT)
):
    raise ValueError("Niepoprawne granice TRAIN/VALIDATION/TEST.")

CATALOG = "workspace"
NLR_TABLE = "workspace.energy_silver.nlr_decision_features_15m"
WEATHER_TABLE = "workspace.energy_silver.open_meteo_forecast"
DAM_TABLE = "workspace.energy_silver.caiso_dam_lmp"
DAY_AHEAD_DATASET_TABLE = (
    "workspace.energy_gold.day_ahead_pv_dataset_hourly"
)
DAY_AHEAD_SCHEDULE_TABLE = (
    "workspace.energy_gold.day_ahead_schedule_hourly"
)

required_tables = {
    "NLR": NLR_TABLE,
    "WEATHER": WEATHER_TABLE,
    "DAM": DAM_TABLE,
}

for source_name, table_name in required_tables.items():
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Brak tabeli {source_name}: {table_name}"
        )
    print(
        f"{source_name}: {spark.table(table_name).count()} rekordów"
    )

print(
    "Zakres:",
    EVALUATION_START_UTC,
    "—",
    EVALUATION_END_UTC,
)
print("Wszystkie wymagane tabele są dostępne.")

###2) Wybór prognozy 72h dostępnej przed zamknięciem DAM

In [0]:
dam_spine_df = (
    spark.table(DAM_TABLE)
    .filter(
        (F.col("interval_start_utc") >= F.lit(EVALUATION_START_UTC))
        & (F.col("interval_start_utc") < F.lit(EVALUATION_END_UTC))
    )
    .select(
        "interval_start_utc",
        "interval_end_utc",
        "delivery_date_local",
        "delivery_hour_local",
        "dam_bid_gate_close_at_utc",
        F.col("lmp_price_usd_mwh").alias(
            "actual_dam_lmp_usd_mwh"
        )
    )
    .dropDuplicates(["interval_start_utc"])
)

weather_candidates_df = (
    spark.table(WEATHER_TABLE)
    .select(
        "forecast_model",
        "forecast_run_id",
        "run_time_utc",
        "available_at_utc",
        "forecast_valid_time_utc",
        "forecast_gti_wm2",
        "temperature_2m_c",
        "relative_humidity_2m_pct",
        "cloud_cover_pct",
        "wind_speed_10m_ms",
        "wind_direction_10m_deg",
        "shortwave_radiation_wm2",
        "is_day"
    )
)

weather_matches_df = (
    dam_spine_df.alias("dam")
    .join(
        weather_candidates_df.alias("weather"),
        (
            F.col("weather.forecast_valid_time_utc")
            == F.col("dam.interval_start_utc")
        )
        &
        (
            F.col("weather.available_at_utc")
            <= F.col("dam.dam_bid_gate_close_at_utc")
        ),
        "left"
    )
    .select(
        "dam.*",
        "weather.*"
    )
)

selection_window = (
    Window
    .partitionBy("interval_start_utc")
    .orderBy(
        F.col("available_at_utc").desc_nulls_last(),
        F.col("run_time_utc").desc_nulls_last()
    )
)

selected_day_ahead_weather_df = (
    weather_matches_df
    .withColumn(
        "_rank",
        F.row_number().over(selection_window)
    )
    .filter(F.col("_rank") == 1)
    .drop("_rank")
    .withColumn(
        "forecast_lead_hours",
        (
            F.col("interval_start_utc").cast("long")
            - F.col("run_time_utc").cast("long")
        ) / 3600.0
    )
)

display(
    selected_day_ahead_weather_df
    .agg(
        F.count("*").alias("row_count"),
        F.sum(
            F.col("forecast_run_id").isNull().cast("int")
        ).alias("missing_forecast_count"),
        F.min("forecast_lead_hours").alias(
            "min_lead_hours"
        ),
        F.max("forecast_lead_hours").alias(
            "max_lead_hours"
        )
    )
)

display(
    selected_day_ahead_weather_df
    .filter(F.col("forecast_run_id").isNull())
    .groupBy("delivery_hour_local")
    .count()
    .orderBy("delivery_hour_local")
)

###3) Dołączenie rzeczywistej produkcji PV

In [0]:
actual_pv_hourly_df = (
    spark.table(NLR_TABLE)
    .filter(
        (
            F.col("target_interval_start_utc")
            >= F.lit(EVALUATION_START_UTC)
        )
        &
        (
            F.col("target_interval_start_utc")
            < F.lit(EVALUATION_END_UTC)
        )
    )
    .withColumn(
        "actual_hour_start_utc",
        F.date_trunc(
            "hour",
            "target_interval_start_utc"
        )
    )
    .groupBy("actual_hour_start_utc")
    .agg(
        F.count("*").alias(
            "pv_15m_interval_count"
        ),
        F.sum(
            "actual_pv_energy_next_15m_mwh"
        ).alias(
            "actual_pv_energy_mwh"
        ),
        F.sum(
            F.col("target_power_is_outlier")
            .cast("int")
        ).alias(
            "pv_outlier_count"
        )
    )
    .withColumn(
        "actual_pv_power_kw",
        F.col("actual_pv_energy_mwh")
        * F.lit(1000.0)
    )
)

day_ahead_dataset_df = (
    selected_day_ahead_weather_df
    .join(
        actual_pv_hourly_df,
        selected_day_ahead_weather_df[
            "interval_start_utc"
        ]
        == actual_pv_hourly_df[
            "actual_hour_start_utc"
        ],
        "left"
    )
    .drop("actual_hour_start_utc")
    .withColumn(
        "is_feature_complete",
        (
            F.col("forecast_run_id").isNotNull()
            &
            F.col("forecast_gti_wm2").isNotNull()
            &
            F.col("temperature_2m_c").isNotNull()
            &
            F.col("cloud_cover_pct").isNotNull()
        )
    )
)

day_ahead_summary_df = (
    day_ahead_dataset_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct(
            "interval_start_utc"
        ).alias("distinct_hour_count"),
        F.sum(
            F.col("forecast_run_id")
            .isNull().cast("int")
        ).alias("missing_forecast_count"),
        F.sum(
            F.col("actual_pv_energy_mwh")
            .isNull().cast("int")
        ).alias("missing_actual_pv_count"),
        F.sum(
            (
                F.col("pv_15m_interval_count") != 4
            ).cast("int")
        ).alias("invalid_interval_count"),
        F.sum("pv_outlier_count").alias(
            "pv_outlier_count"
        ),
        F.sum(
            F.col("is_feature_complete")
            .cast("int")
        ).alias("feature_complete_count"),
        F.sum(
            (
                F.col("available_at_utc")
                > F.col("dam_bid_gate_close_at_utc")
            ).cast("int")
        ).alias("availability_violation_count")
    )
)

display(day_ahead_summary_df)

###4) Cechy, podział czasowy i zapis tabeli

In [0]:
PI = 3.141592653589793

day_ahead_model_dataset_df = (
    day_ahead_dataset_df
    .withColumn(
        "delivery_hour_sin",
        F.sin(
            F.lit(2.0 * PI)
            * F.col("delivery_hour_local")
            / F.lit(24.0)
        )
    )
    .withColumn(
        "delivery_hour_cos",
        F.cos(
            F.lit(2.0 * PI)
            * F.col("delivery_hour_local")
            / F.lit(24.0)
        )
    )
    .withColumn(
        "delivery_day_of_week",
        F.pmod(
            F.dayofweek("delivery_date_local")
            + F.lit(5),
            F.lit(7)
        )
    )
    .withColumn(
        "delivery_day_of_week_sin",
        F.sin(
            F.lit(2.0 * PI)
            * F.col("delivery_day_of_week")
            / F.lit(7.0)
        )
    )
    .withColumn(
        "delivery_day_of_week_cos",
        F.cos(
            F.lit(2.0 * PI)
            * F.col("delivery_day_of_week")
            / F.lit(7.0)
        )
    )
    .withColumn(
        "forecast_wind_direction_sin",
        F.sin(
            F.radians("wind_direction_10m_deg")
        )
    )
    .withColumn(
        "forecast_wind_direction_cos",
        F.cos(
            F.radians("wind_direction_10m_deg")
        )
    )
    .withColumn(
        "dataset_split",
        F.when(
            F.col("delivery_date_local")
            < F.lit(TRAIN_END_DATE_TEXT).cast("date"),
            F.lit("TRAIN")
        )
        .when(
            F.col("delivery_date_local")
            < F.lit(VALIDATION_END_DATE_TEXT).cast("date"),
            F.lit("VALIDATION")
        )
        .otherwise(F.lit("TEST"))
    )
    .withColumn(
        "is_model_ready",
        (
            F.col("is_feature_complete")
            &
            F.col("actual_pv_energy_mwh").isNotNull()
        )
    )
    .withColumn(
        "_dataset_created_at_utc",
        F.current_timestamp()
    )
)

(
    day_ahead_model_dataset_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DAY_AHEAD_DATASET_TABLE)
)

saved_day_ahead_df = spark.table(
    DAY_AHEAD_DATASET_TABLE
)

print(f"Zapisano tabelę: {DAY_AHEAD_DATASET_TABLE}")
print(f"Liczba rekordów: {saved_day_ahead_df.count()}")

display(
    saved_day_ahead_df
    .groupBy("dataset_split")
    .agg(
        F.count("*").alias("row_count"),
        F.sum(
            F.col("is_model_ready").cast("int")
        ).alias("model_ready_count"),
        F.min("interval_start_utc").alias(
            "min_delivery_time_utc"
        ),
        F.max("interval_start_utc").alias(
            "max_delivery_time_utc"
        )
    )
    .orderBy("min_delivery_time_utc")
)

###5) Modele day-ahead i ocena VALIDATION

In [0]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

DAY_AHEAD_FEATURE_COLUMNS = [
    "forecast_gti_wm2",
    "temperature_2m_c",
    "relative_humidity_2m_pct",
    "cloud_cover_pct",
    "wind_speed_10m_ms",
    "shortwave_radiation_wm2",
    "is_day",
    "forecast_lead_hours",
    "delivery_hour_sin",
    "delivery_hour_cos",
    "delivery_day_of_week_sin",
    "delivery_day_of_week_cos",
    "forecast_wind_direction_sin",
    "forecast_wind_direction_cos",
]

day_ahead_pdf = (
    spark.table(
        "workspace.energy_gold."
        "day_ahead_pv_dataset_hourly"
    )
    .filter("is_model_ready = true")
    .orderBy("interval_start_utc")
    .toPandas()
)

train_pdf = day_ahead_pdf[
    day_ahead_pdf["dataset_split"] == "TRAIN"
].copy()

validation_pdf = day_ahead_pdf[
    day_ahead_pdf["dataset_split"] == "VALIDATION"
].copy()

X_train = train_pdf[DAY_AHEAD_FEATURE_COLUMNS]
y_train = train_pdf["actual_pv_power_kw"]

X_validation = validation_pdf[
    DAY_AHEAD_FEATURE_COLUMNS
]
y_validation = validation_pdf[
    "actual_pv_power_kw"
]

day_ahead_models = {
    "GTI_LINEAR": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            LinearRegression(positive=True)
        ),
    ]),
    "DAY_AHEAD_PV_RIDGE": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0)),
    ]),
    "DAY_AHEAD_PV_HGB": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            HistGradientBoostingRegressor(
                learning_rate=0.05,
                max_iter=200,
                max_leaf_nodes=15,
                l2_regularization=1.0,
                random_state=42
            )
        ),
    ]),
}

validation_results = []
DAY_AHEAD_MODEL_FEATURES = {
    "GTI_LINEAR": [
        "forecast_gti_wm2"
    ],
    "DAY_AHEAD_PV_RIDGE":
        DAY_AHEAD_FEATURE_COLUMNS,
    "DAY_AHEAD_PV_HGB":
        DAY_AHEAD_FEATURE_COLUMNS,
}

validation_results = []
validation_predictions = {}

for model_name, model in day_ahead_models.items():
    feature_columns = DAY_AHEAD_MODEL_FEATURES[
        model_name
    ]

    X_train_model = train_pdf[
        feature_columns
    ]

    X_validation_model = validation_pdf[
        feature_columns
    ]

    model.fit(
        X_train_model,
        y_train
    )

    predictions = model.predict(
        X_validation_model
    )

    predictions = np.where(
        validation_pdf["is_day"].to_numpy() == 0,
        0.0,
        np.maximum(predictions, 0.0)
    )

    validation_predictions[model_name] = predictions

    validation_results.append({
        "model_name": model_name,
        "dataset_split": "VALIDATION",
        "mae": mean_absolute_error(
            y_validation,
            predictions
        ),
        "rmse": np.sqrt(
            mean_squared_error(
                y_validation,
                predictions
            )
        ),
        "r2": r2_score(
            y_validation,
            predictions
        ),
        "mean_error": float(
            np.mean(predictions - y_validation)
        ),
        "feature_count": len(feature_columns),
    })

validation_results_pdf = (
    pd.DataFrame(validation_results)
    .sort_values(["mae", "rmse"])
    .reset_index(drop=True)
)

print(f"TRAIN: {len(train_pdf)}")
print(f"VALIDATION: {len(validation_pdf)}")
print(
    f"Liczba pełnego zestawu cech: "
    f"{len(DAY_AHEAD_FEATURE_COLUMNS)}"
)

display(validation_results_pdf)

###6) Finalny TEST day-ahead

In [0]:
from sklearn.base import clone

train_validation_pdf = pd.concat(
    [train_pdf, validation_pdf],
    ignore_index=True
)

test_pdf = day_ahead_pdf[
    day_ahead_pdf["dataset_split"] == "TEST"
].copy()

X_train_validation = train_validation_pdf[
    DAY_AHEAD_FEATURE_COLUMNS
]

y_train_validation = train_validation_pdf[
    "actual_pv_power_kw"
]

X_test = test_pdf[
    DAY_AHEAD_FEATURE_COLUMNS
]

y_test = test_pdf[
    "actual_pv_power_kw"
]

final_day_ahead_pv_model = clone(
    day_ahead_models["DAY_AHEAD_PV_HGB"]
)

final_day_ahead_pv_model.fit(
    X_train_validation,
    y_train_validation
)

test_predictions = final_day_ahead_pv_model.predict(
    X_test
)

test_predictions = np.where(
    test_pdf["is_day"].to_numpy() == 0,
    0.0,
    np.maximum(test_predictions, 0.0)
)

test_metrics_pdf = pd.DataFrame([{
    "model_name": "DAY_AHEAD_PV_HGB_FINAL",
    "dataset_split": "TEST",
    "mae": mean_absolute_error(
        y_test,
        test_predictions
    ),
    "rmse": np.sqrt(
        mean_squared_error(
            y_test,
            test_predictions
        )
    ),
    "r2": r2_score(
        y_test,
        test_predictions
    ),
    "mean_error": float(
        np.mean(test_predictions - y_test)
    ),
    "row_count": len(test_pdf),
}])

day_ahead_test_predictions_pdf = (
    test_pdf[[
        "interval_start_utc",
        "interval_end_utc",
        "delivery_date_local",
        "delivery_hour_local",
        "dam_bid_gate_close_at_utc",
        "forecast_run_id",
        "run_time_utc",
        "available_at_utc",
        "forecast_lead_hours",
        "actual_pv_power_kw",
        "actual_pv_energy_mwh",
        "actual_dam_lmp_usd_mwh",
    ]]
    .copy()
)

day_ahead_test_predictions_pdf[
    "predicted_pv_power_kw"
] = test_predictions

day_ahead_test_predictions_pdf[
    "predicted_pv_energy_mwh"
] = test_predictions / 1000.0

day_ahead_test_predictions_pdf[
    "recommended_dam_schedule_mwh"
] = (
    day_ahead_test_predictions_pdf[
        "predicted_pv_energy_mwh"
    ]
)

display(test_metrics_pdf)

print(
    "Finalna ocena TEST wykonana jednokrotnie."
)
print(
    "Liczba harmonogramów godzinowych:",
    len(day_ahead_test_predictions_pdf)
)

###7) Zapis harmonogramu DAM do Delta

In [0]:
from pyspark.sql import functions as F

DAY_AHEAD_SCHEDULE_TABLE = (
    "workspace.energy_gold."
    "day_ahead_schedule_hourly"
)
POI_EXPORT_LIMIT_MW = 0.8

day_ahead_schedule_spark_df = (
    spark.createDataFrame(
        day_ahead_test_predictions_pdf
    )
    .withColumn(
        "system_id",
        F.lit(2107).cast("int")
    )
    .withColumn(
        "recommended_dam_schedule_mwh",
        F.least(
            F.greatest(
                F.col(
                    "recommended_dam_schedule_mwh"
                ),
                F.lit(0.0)
            ),
            F.lit(POI_EXPORT_LIMIT_MW)
        )
    )
    .withColumn(
        "schedule_error_mwh",
        F.col("actual_pv_energy_mwh")
        - F.col(
            "recommended_dam_schedule_mwh"
        )
    )
    .withColumn(
        "absolute_schedule_error_mwh",
        F.abs("schedule_error_mwh")
    )
    .withColumn(
        "dam_schedule_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("system_id").cast("string"),
                F.col(
                    "interval_start_utc"
                ).cast("string")
            ),
            256
        )
    )
    .withColumn(
        "pv_forecast_model",
        F.lit("DAY_AHEAD_PV_HGB_FINAL")
    )
    .withColumn(
        "poi_export_limit_mw",
        F.lit(POI_EXPORT_LIMIT_MW)
    )
    .withColumn(
        "_schedule_created_at_utc",
        F.current_timestamp()
    )
)

(
    day_ahead_schedule_spark_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DAY_AHEAD_SCHEDULE_TABLE)
)

saved_schedule_df = spark.table(
    DAY_AHEAD_SCHEDULE_TABLE
)

print(f"Zapisano tabelę: {DAY_AHEAD_SCHEDULE_TABLE}")

display(
    saved_schedule_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct(
            "dam_schedule_id"
        ).alias("distinct_schedule_count"),
        F.sum(
            (
                F.col("recommended_dam_schedule_mwh")
                < 0
            ).cast("int")
        ).alias("negative_schedule_count"),
        F.sum(
            (
                F.col("recommended_dam_schedule_mwh")
                > F.col("poi_export_limit_mw")
            ).cast("int")
        ).alias("export_limit_violation_count"),
        F.avg(
            "absolute_schedule_error_mwh"
        ).alias("average_absolute_error_mwh")
    )
)

###8) MLflow dla modelu day-ahead

In [0]:
import mlflow
import mlflow.sklearn

from mlflow.models import infer_signature

input_example = X_test.head(5).copy()

# Zapobiega ostrzeżeniu MLflow o integer.
input_example["is_day"] = (
    input_example["is_day"].astype(float)
)

example_predictions = (
    final_day_ahead_pv_model.predict(
        input_example
    )
)

model_signature = infer_signature(
    input_example,
    example_predictions
)

metric_row = test_metrics_pdf.iloc[0]

if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(
    run_name="day_ahead_pv_hgb_final_v1"
) as run:

    mlflow.log_params({
        "model_name":
            "DAY_AHEAD_PV_HGB_FINAL",
        "dataset_split": "TEST",
        "feature_count":
            len(DAY_AHEAD_FEATURE_COLUMNS),
        "test_row_count": len(test_pdf),
        "forecast_type": "DAY_AHEAD",
        "poi_export_limit_mw": 0.8
    })

    mlflow.log_metrics({
        "test_mae_kw":
            float(metric_row["mae"]),
        "test_rmse_kw":
            float(metric_row["rmse"]),
        "test_r2":
            float(metric_row["r2"]),
        "test_mean_error_kw":
            float(metric_row["mean_error"])
    })

    mlflow.set_tags({
        "project": "energy_market_replay",
        "target": "hourly_pv_power",
        "source_systems":
            "NLR_OPEN_METEO_CAISO"
    })

    logged_day_ahead_model = (
        mlflow.sklearn.log_model(
            sk_model=final_day_ahead_pv_model,
            name="day_ahead_pv_hgb_final",
            input_example=input_example,
            signature=model_signature
        )
    )

    day_ahead_run_id = run.info.run_id

print(f"Run ID: {day_ahead_run_id}")
print(
    "Model URI:",
    logged_day_ahead_model.model_uri
)

###9) Rejestracja w Unity Catalog

In [0]:
import mlflow

from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")

if "logged_day_ahead_model" not in globals():
    raise NameError(
        "Brak modelu z bieżącego runu. Uruchom notebook od początku."
    )

REGISTERED_MODEL_NAME = (
    "workspace.energy_gold.day_ahead_pv_hgb_hourly"
)

registered_version = mlflow.register_model(
    model_uri=logged_day_ahead_model.model_uri,
    name=REGISTERED_MODEL_NAME,
)

client = MlflowClient(registry_uri="databricks-uc")
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="Champion",
    version=str(registered_version.version),
)

print(f"Model: {REGISTERED_MODEL_NAME}")
print(f"Wersja: {registered_version.version}")
print("Alias: Champion")